In [4]:
#using pyspark instead of pandas because its more efficient for large datasets and distributed computing.
from pyspark.sql import SparkSession

In [5]:
spark = SparkSession.builder.appName("mock").getOrCreate()

In [6]:
df = spark.read.csv("mock.csv", header=True, inferSchema=True)

In [7]:
df.printSchema()

root
 |-- student_id: string (nullable = true)
 |-- student_name: string (nullable = true)
 |-- level: string (nullable = true)
 |-- department: string (nullable = true)
 |-- course_code: string (nullable = true)
 |-- course_name: string (nullable = true)
 |-- week: integer (nullable = true)
 |-- classes_held: integer (nullable = true)
 |-- classes_attended: integer (nullable = true)
 |-- attendance_percent: double (nullable = true)
 |-- assignment_submitted: integer (nullable = true)
 |-- assignment_score: integer (nullable = true)
 |-- quiz_score: integer (nullable = true)
 |-- lms_logins: integer (nullable = true)
 |-- helpdesk_requests: integer (nullable = true)
 |-- last_login_days_ago: integer (nullable = true)
 |-- risk_points: integer (nullable = true)
 |-- risk_level: string (nullable = true)
 |-- risk_label: integer (nullable = true)
 |-- mentor_review_needed: integer (nullable = true)
 |-- peer_tutorial_recommended: integer (nullable = true)
 |-- risk_reason: string (nullabl

In [8]:
df.describe().show()

+-------+----------+------------+------+----------------+-----------+--------------------+------------------+------------------+------------------+------------------+--------------------+------------------+------------------+-----------------+------------------+-------------------+-----------------+----------+------------------+--------------------+-------------------------+--------------------+
|summary|student_id|student_name| level|      department|course_code|         course_name|              week|      classes_held|  classes_attended|attendance_percent|assignment_submitted|  assignment_score|        quiz_score|       lms_logins| helpdesk_requests|last_login_days_ago|      risk_points|risk_level|        risk_label|mentor_review_needed|peer_tutorial_recommended|         risk_reason|
+-------+----------+------------+------+----------------+-----------+--------------------+------------------+------------------+------------------+------------------+--------------------+---------------

In [9]:
df.columns

['student_id',
 'student_name',
 'level',
 'department',
 'course_code',
 'course_name',
 'week',
 'classes_held',
 'classes_attended',
 'attendance_percent',
 'assignment_submitted',
 'assignment_score',
 'quiz_score',
 'lms_logins',
 'helpdesk_requests',
 'last_login_days_ago',
 'risk_points',
 'risk_level',
 'risk_label',
 'mentor_review_needed',
 'peer_tutorial_recommended',
 'risk_reason']

In [10]:
from pyspark.ml.feature import VectorAssembler

In [11]:
assembler = VectorAssembler(inputCols=[
    "attendance_percent",
    "assignment_submitted",
    "assignment_score",
    "quiz_score",
    "lms_logins",
    "helpdesk_requests",
    "last_login_days_ago"], outputCol="features")

In [12]:
output = assembler.transform(df)

In [13]:
df_final = output.select("features", "risk_label")

In [14]:
df_final.show()

+--------------------+----------+
|            features|risk_label|
+--------------------+----------+
|[100.0,0.0,0.0,89...|         1|
|(7,[3,5,6],[84.0,...|         1|
|[50.0,1.0,35.0,40...|         1|
|[40.0,0.0,0.0,68....|         1|
|[50.0,1.0,81.0,78...|         1|
|[0.0,1.0,84.0,57....|         1|
|[57.1,1.0,88.0,66...|         0|
|[83.3,1.0,76.0,29...|         1|
|[37.5,1.0,88.0,27...|         1|
|[57.1,0.0,0.0,92....|         1|
|[100.0,1.0,51.0,9...|         0|
|[20.0,1.0,40.0,26...|         1|
|[14.3,1.0,73.0,79...|         1|
|[50.0,1.0,64.0,20...|         1|
|[12.5,1.0,54.0,84...|         0|
|[100.0,1.0,73.0,6...|         0|
|[12.5,0.0,0.0,82....|         1|
|[100.0,1.0,70.0,4...|         1|
|[50.0,1.0,76.0,67...|         1|
|[20.0,0.0,0.0,22....|         1|
+--------------------+----------+
only showing top 20 rows


In [15]:
train, test = df_final.randomSplit([0.7, 0.3], seed=42)

In [16]:
from pyspark.ml.classification import LogisticRegression

In [17]:
lr = LogisticRegression(labelCol='risk_label')

In [18]:
lrm = lr.fit(train)

In [19]:
lrm_summary = lrm.summary

In [20]:
lrm_summary.predictions.show()

+--------------------+----------+--------------------+--------------------+----------+
|            features|risk_label|       rawPrediction|         probability|prediction|
+--------------------+----------+--------------------+--------------------+----------+
|(7,[0,3,4],[66.7,...|       1.0|[0.54372018419453...|[0.63267740223734...|       0.0|
|(7,[3,4,6],[92.0,...|       1.0|[-4.2789731468328...|[0.01366749488046...|       1.0|
|(7,[3,4,6],[97.0,...|       1.0|[-2.7941575261151...|[0.05764070993163...|       1.0|
|(7,[3,5,6],[84.0,...|       1.0|[-8.7543386676730...|[1.57750414253891...|       1.0|
|[0.0,0.0,0.0,59.0...|       1.0|[-12.082396886685...|[5.65821240627205...|       1.0|
|[0.0,1.0,38.0,64....|       1.0|[-3.7378247230629...|[0.02325229101186...|       1.0|
|[0.0,1.0,49.0,42....|       1.0|[-4.9056530542299...|[0.00735018070305...|       1.0|
|[0.0,1.0,54.0,85....|       1.0|[-4.2515055104056...|[0.01404276713935...|       1.0|
|[0.0,1.0,60.0,66....|       1.0|[-8.376489

In [21]:
lrm_summary.predictions.describe().show()

+-------+-------------------+------------------+
|summary|         risk_label|        prediction|
+-------+-------------------+------------------+
|  count|                136|               136|
|   mean| 0.7058823529411765|0.7058823529411765|
| stddev|0.45732956038002365|0.4573295603800236|
|    min|                0.0|               0.0|
|    max|                1.0|               1.0|
+-------+-------------------+------------------+



In [22]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

In [23]:
pred_labels = lrm.evaluate(test)

In [24]:
pred_labels.predictions.show()

+--------------------+----------+--------------------+--------------------+----------+
|            features|risk_label|       rawPrediction|         probability|prediction|
+--------------------+----------+--------------------+--------------------+----------+
|(7,[3,4,6],[93.0,...|         1|[-7.7332726006122...|[4.37816575342202...|       1.0|
|[0.0,0.0,0.0,62.0...|         1|[-16.016030452002...|[1.10745555168161...|       1.0|
|[0.0,1.0,39.0,85....|         1|[-2.5173581190709...|[0.07465023495098...|       1.0|
|[0.0,1.0,48.0,75....|         1|[-4.9429999617103...|[0.00708264532186...|       1.0|
|[0.0,1.0,65.0,23....|         1|[-13.569151954541...|[1.27935663679099...|       1.0|
|[0.0,1.0,65.0,68....|         1|[-8.0825403285922...|[3.08789986874300...|       1.0|
|[0.0,1.0,67.0,53....|         1|[-9.6542825348946...|[6.4146137247926E...|       1.0|
|[0.0,1.0,77.0,70....|         1|[-3.2634865709552...|[0.03684527889755...|       1.0|
|[0.0,1.0,78.0,50....|         1|[-6.539368

In [25]:
eval = BinaryClassificationEvaluator(rawPredictionCol="rawPrediction", labelCol="risk_label")

In [26]:
auc = eval.evaluate(pred_labels.predictions)

In [27]:
auc

0.9978354978354979

In [30]:
final_model = lr.fit(df_final)

In [32]:
res = final_model.transform(df_final)

In [34]:
res.select( "risk_label", "prediction").show()

+----------+----------+
|risk_label|prediction|
+----------+----------+
|         1|       1.0|
|         1|       1.0|
|         1|       1.0|
|         1|       1.0|
|         1|       1.0|
|         1|       1.0|
|         0|       0.0|
|         1|       1.0|
|         1|       1.0|
|         1|       1.0|
|         0|       0.0|
|         1|       1.0|
|         1|       1.0|
|         1|       1.0|
|         0|       0.0|
|         0|       0.0|
|         1|       1.0|
|         1|       1.0|
|         1|       1.0|
|         1|       1.0|
+----------+----------+
only showing top 20 rows


In [39]:
eval_df = res.select("risk_label", "prediction")
tp = eval_df.filter("risk_label = 1 AND prediction = 1").count()
tn = eval_df.filter("risk_label = 0 AND prediction = 0").count()
fp = eval_df.filter("risk_label = 0 AND prediction = 1").count()
fn = eval_df.filter("risk_label = 1 AND prediction = 0").count()
accuracy = (tp + tn) / (tp + tn + fp + fn)
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0

print(f"Accuracy: {accuracy}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")

Accuracy: 0.9523809523809523
Precision: 0.9708029197080292
Recall: 0.9637681159420289
